In [11]:
import sys
from pathlib import Path

# Get the backend root directory
PROJECT_ROOT = Path.cwd().parent

# Add backend root to Python import path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("App directory exists:", (PROJECT_ROOT / "app").exists())

Project root: c:\Users\hp\personal\HireMatch\hirematch-backend
App directory exists: True


In [8]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

cv_df = pd.read_csv(PROCESSED_DIR / "cv_processed.csv")
job_df = pd.read_csv(PROCESSED_DIR / "job_processed.csv")
matches_df = pd.read_csv(RAW_DIR / "matches.csv")

print("CV:", cv_df.shape)
print("Jobs:", job_df.shape)
print("Matches:", matches_df.shape)

print("CV dataset:", cv_df.shape)
print("Job dataset:", job_df.shape)
print("Matches dataset:", matches_df.shape)

print("\nSample CV skills:")
print(cv_df["skills_list"].iloc[0])
print("Data type:", type(cv_df["skills_list"].iloc[0]))

print("\nSample Job skills:")
print(job_df["skills_list"].iloc[0])
print("Data type:", type(job_df["skills_list"].iloc[0]))

CV: (9544, 8)
Jobs: (1167, 8)
Matches: (122560, 13)
CV dataset: (9544, 8)
Job dataset: (1167, 8)
Matches dataset: (122560, 13)

Sample CV skills:
["big data", "hadoop", "hive", "python", "mapreduce", "spark", "java", "machine learning", "cloud", "hdfs", "yarn", "core java", "data science", "c++", "data structures", "dbms", "rdbms", "informatica", "talend", "amazon redshift", "microsoft azure"]
Data type: <class 'str'>

Sample Job skills:
["financial management", "accounting", "gaap principles", "budgeting", "forecasting", "financial statement preparation", "cash flow management", "accounts payable", "internal controls", "risk management", "microsoft excel", "microsoft office suite", "negotiating skills", "attention to detail", "self-motivated", "results oriented", "confidentiality", "ethics and integrity", "investment oversight", "pension administration", "raiser s edge software", "pos systems", "consensus building"]
Data type: <class 'str'>


In [9]:
import json
import pandas as pd


def parse_skills(value):
    """Convert JSON-formatted skill strings into Python lists."""

    if isinstance(value, list):
        return value

    if not isinstance(value, str) or not value.strip():
        return []

    try:
        skills = json.loads(value)

        if isinstance(skills, list):
            return skills

        return []

    except json.JSONDecodeError:
        return []


cv_df["skills_list"] = cv_df["skills_list"].apply(parse_skills)
job_df["skills_list"] = job_df["skills_list"].apply(parse_skills)


print("CV skills type:", type(cv_df["skills_list"].iloc[0]))
print("Job skills type:", type(job_df["skills_list"].iloc[0]))

print("\nSample CV skills:")
print(cv_df["skills_list"].iloc[0])

print("\nSample Job skills:")
print(job_df["skills_list"].iloc[0])

print("\nCV records with empty skill lists:",
      cv_df["skills_list"].apply(len).eq(0).sum())

print("Job records with empty skill lists:",
      job_df["skills_list"].apply(len).eq(0).sum())

CV skills type: <class 'list'>
Job skills type: <class 'list'>

Sample CV skills:
['big data', 'hadoop', 'hive', 'python', 'mapreduce', 'spark', 'java', 'machine learning', 'cloud', 'hdfs', 'yarn', 'core java', 'data science', 'c++', 'data structures', 'dbms', 'rdbms', 'informatica', 'talend', 'amazon redshift', 'microsoft azure']

Sample Job skills:
['financial management', 'accounting', 'gaap principles', 'budgeting', 'forecasting', 'financial statement preparation', 'cash flow management', 'accounts payable', 'internal controls', 'risk management', 'microsoft excel', 'microsoft office suite', 'negotiating skills', 'attention to detail', 'self-motivated', 'results oriented', 'confidentiality', 'ethics and integrity', 'investment oversight', 'pension administration', 'raiser s edge software', 'pos systems', 'consensus building']

CV records with empty skill lists: 20
Job records with empty skill lists: 0


In [12]:
from app.ml.skill_matcher import SkillMatcher

matcher = SkillMatcher()

# Select a manageable sample of 2,000 real matched pairs
sample_matches = matches_df.sample(
    n=2000,
    random_state=42
).copy()

# Create lookup dictionaries for faster access
cv_skills_map = cv_df.set_index("candidate_id")["skills_list"].to_dict()
job_skills_map = job_df.set_index("job_id")["skills_list"].to_dict()


# Calculate skill matching for every sampled pair
results = []

for _, row in sample_matches.iterrows():

    candidate_id = row["candidate_id"]
    job_id = row["job_id"]

    resume_skills = cv_skills_map.get(candidate_id, [])
    required_skills = job_skills_map.get(job_id, [])

    result = matcher.calculate_skill_match(
        resume_skills,
        required_skills
    )

    results.append({
        "candidate_id": candidate_id,
        "job_id": job_id,
        "skill_coverage": result["skill_coverage"],
        "matched_count": result["matched_count"],
        "missing_count": result["missing_count"],
        "total_required_skills": result["total_required_skills"]
    })


skill_results_df = pd.DataFrame(results)

print("Evaluated pairs:", len(skill_results_df))
print("\nFirst 5 results:")
display(skill_results_df.head())

Evaluated pairs: 2000

First 5 results:


,candidate_id,job_id,skill_coverage,matched_count,missing_count,total_required_skills
0,cand_00302,3901981623,0.00,0,14,14
1,cand_03670,3901393491,18.18,2,9,11
2,cand_06785,3895849293,0.00,0,37,37
3,cand_00200,3902329403,0.00,0,8,8
4,cand_08451,3905350188,16.67,2,10,12


In [13]:
print("Skill Coverage Statistics")
print("-------------------------")

print(
    skill_results_df["skill_coverage"].describe()
)

print("\nAverage skill coverage:")
print(
    round(skill_results_df["skill_coverage"].mean(), 2),
    "%"
)

print("\nPairs with zero skill coverage:")
print(
    (skill_results_df["skill_coverage"] == 0).sum()
)

print("\nPairs with 100% skill coverage:")
print(
    (skill_results_df["skill_coverage"] == 100).sum()
)

Skill Coverage Statistics
-------------------------
count    2000.000000
mean        8.482740
std         9.960475
min         0.000000
25%         0.000000
50%         6.250000
75%        13.330000
max        57.140000
Name: skill_coverage, dtype: float64

Average skill coverage:
8.48 %

Pairs with zero skill coverage:
816

Pairs with 100% skill coverage:
0


In [14]:
print("Result columns:")
print(skill_results_df.columns.tolist())

print("\nFirst 5 rows:")
print(skill_results_df.head().to_string(index=False))

print("\nColumn data types:")
print(skill_results_df.dtypes)

print("\nCandidate IDs found in CV dataset:",
      skill_results_df["candidate_id"].isin(cv_df["candidate_id"]).all())

print("Job IDs found in job dataset:",
      skill_results_df["job_id"].isin(job_df["job_id"]).all())

Result columns:
['candidate_id', 'job_id', 'skill_coverage', 'matched_count', 'missing_count', 'total_required_skills']

First 5 rows:
candidate_id     job_id  skill_coverage  matched_count  missing_count  total_required_skills
  cand_00302 3901981623            0.00              0             14                     14
  cand_03670 3901393491           18.18              2              9                     11
  cand_06785 3895849293            0.00              0             37                     37
  cand_00200 3902329403            0.00              0              8                      8
  cand_08451 3905350188           16.67              2             10                     12

Column data types:
candidate_id                 str
job_id                     int64
skill_coverage           float64
matched_count              int64
missing_count              int64
total_required_skills      int64
dtype: object

Candidate IDs found in CV dataset: True
Job IDs found in job dataset: True

In [15]:
sample = sample_matches.iloc[0]

candidate_id = sample["candidate_id"]
job_id = sample["job_id"]

resume_skills = cv_skills_map[candidate_id]
required_skills = job_skills_map[job_id]

result = matcher.calculate_skill_match(
    resume_skills,
    required_skills
)

print("Candidate ID:", candidate_id)
print("Job ID:", job_id)

print("\nCandidate skills:")
print(resume_skills)

print("\nRequired job skills:")
print(required_skills)

print("\nMatched skills:")
print(result["matched_skills"])

print("\nMissing skills:")
print(result["missing_skills"])

print("\nSkill coverage:", result["skill_coverage"], "%")

Candidate ID: cand_00302
Job ID: 3901981623

Candidate skills:
['data scientist', 'data analyst', 'software development', 'machine learning', 'natural language processing', 'business solutions', 'process flow', 'logistic regression', 'mysql', 'mongodb', 'python', 'powerbi', 'hadoop', 'asp.net mvc strong understanding of database design', 'database administrator dba', 'database management', 'elasticsearch', 'mysql database', 'nosql database', 'redis', 'data science']

Required job skills:
['network infrastructure', 'cat5/6 cabling', 'domain controller', 'active directory', 'management and implementations', 'hands-on support', 'scalable solutions', 'wireless networks', 'network switches', 'wan/lan', 'pos systems', 'msp experience', 'firewall applications', 'vpn applications']

Matched skills:
[]

Missing skills:
['network infrastructure', 'cat5/6 cabling', 'domain controller', 'active directory', 'management and implementations', 'hands-on support', 'scalable solutions', 'wireless networ

In [16]:
cv_df["candidate_id"] = cv_df["candidate_id"].astype(str)
job_df["job_id"] = job_df["job_id"].astype(str)

matches_df["candidate_id"] = matches_df["candidate_id"].astype(str)
matches_df["job_id"] = matches_df["job_id"].astype(str)

print("CV candidate ID type:",
      cv_df["candidate_id"].dtype)

print("Job ID type:",
      job_df["job_id"].dtype)

print("Match job ID type:",
      matches_df["job_id"].dtype)

CV candidate ID type: str
Job ID type: str
Match job ID type: str


In [17]:
cv_skills_map = cv_df.set_index("candidate_id")["skills_list"].to_dict()
job_skills_map = job_df.set_index("job_id")["skills_list"].to_dict()

In [18]:
print("Total evaluated pairs:", len(skill_results_df))

print("\nCoverage range:")
print(
    skill_results_df["skill_coverage"].min(),
    "to",
    skill_results_df["skill_coverage"].max()
)

print("\nAverage coverage:",
      round(skill_results_df["skill_coverage"].mean(), 2))

print("\nAll coverage values valid:",
      skill_results_df["skill_coverage"].between(0, 100).all())

Total evaluated pairs: 2000

Coverage range:
0.0 to 57.14

Average coverage: 8.48

All coverage values valid: True
